<div style='background:#0f172a;padding:28px 32px;border-radius:12px;color:#e2e8f0;font-family:system-ui,Arial'>
<h1 style='margin:0;font-size:22px'>Segmentación + Visualización — BraTS 2024 GLI</h1>
<p style='color:#94a3b8;margin:8px 0 0'>Limpieza (Wiener + N4 + normalización) → 4 métodos clásicos (Otsu multinivel, GMM, crecimiento de regiones, watershed) → métricas Dice/Jaccard vs ground truth → mosaicos 3 vistas y tabla comparativa.<br>
Objetivo: <b>tumor completo (whole tumor)</b> en <b>FLAIR (t2f)</b>. Sin deep learning.</p></div>

In [ ]:
# === Setup ===
import importlib, subprocess, sys
for pkg, mod in [('SimpleITK','SimpleITK'), ('nibabel','nibabel'), ('scikit-image','skimage'), ('scikit-learn','sklearn')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'])
import os, glob, gc, json, shutil, zipfile, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import nibabel as nib, SimpleITK as sitk
from scipy.signal import wiener
from scipy import ndimage as ndi
from skimage.filters import threshold_multiotsu, sobel
from skimage.segmentation import watershed
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
print('Setup OK | SimpleITK', sitk.Version.VersionString())

In [ ]:
# === Fuente de datos (Drive) — robusta ===
try:
    from google.colab import drive
    drive.mount('/content/drive'); EN_COLAB = True
except Exception:
    EN_COLAB = False
if EN_COLAB:
    try: os.listdir('/content/drive/MyDrive')
    except Exception:
        from google.colab import drive; drive.mount('/content/drive', force_remount=True)
DRIVE_DIR = '/content/drive/MyDrive/BRATS-2024'   # >>> AJUSTA SI HACE FALTA <<<
def _es_entrenamiento(z):
    n=os.path.basename(z).lower(); return ('training' in n) and ('validation' not in n)
def _encontrar_zip(d):
    cands = sorted(glob.glob(os.path.join(d,'*.zip')))
    if not cands and os.path.isdir('/content/drive/MyDrive'):
        cands = sorted(glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True))
    ent=[z for z in cands if _es_entrenamiento(z)]
    if ent:
        principal=[z for z in ent if 'additional' not in os.path.basename(z).lower()]
        return (principal or ent)[0]
    return cands[0] if cands else None
TRAINING_ZIP=_encontrar_zip(DRIVE_DIR)
print('ZIP:', TRAINING_ZIP)
if TRAINING_ZIP is None: raise FileNotFoundError('No se encontró el ZIP de entrenamiento en Drive.')
DRIVE_BASE=os.path.join(DRIVE_DIR,'final-project') if os.path.isdir(DRIVE_DIR) else None
EDA_DIR=os.path.join(DRIVE_BASE,'eda') if DRIVE_BASE else 'eda'   # entradas del EDA (en Drive)
# Salidas a disco LOCAL: escribir muchos .npy directo en Drive provoca FileNotFoundError intermitentes
# (inconsistencia del FUSE). Se trabaja en /content y al final se copia a Drive + se descarga el ZIP.
OUT_BASE='/content/final-project-out'
LIMP_DIR=os.path.join(OUT_BASE,'limpieza'); SEG_DIR=os.path.join(OUT_BASE,'segmentacion'); VIS_DIR=os.path.join(OUT_BASE,'visualizacion')
for d in [LIMP_DIR,SEG_DIR,VIS_DIR]: os.makedirs(d,exist_ok=True)
TMP_DIR='/content/_seg_tmp'; os.makedirs(TMP_DIR,exist_ok=True)
def _buscar_eda(n):
    p=os.path.join(EDA_DIR,n)
    if os.path.exists(p): return p
    h=glob.glob(os.path.join(DRIVE_DIR,'**',n),recursive=True); return h[0] if h else None
print('EDA:', EDA_DIR, '| salidas seg:', SEG_DIR, '| vis:', VIS_DIR)

In [ ]:
# === Entradas del EDA + configuración ===
CASOS_CSV=_buscar_eda('casos_demostrativos.csv'); OTSU_CSV=_buscar_eda('EDA_intensidad_otsu.csv')
df_demo=pd.read_csv(CASOS_CSV) if CASOS_CSV else pd.DataFrame()
CASOS=df_demo['case_id'].astype(str).tolist() if len(df_demo) else []
if not CASOS: raise RuntimeError('No hay casos_demostrativos.csv del EDA.')

MOD_WT='t2f'                 # FLAIR: el tumor completo es hiperintenso
LABEL_MAP={1:'NETC',2:'SNFH',3:'ET',4:'RC'}
METODOS=['otsu','gmm','crecimiento','watershed']
# Limpieza
WIENER_SIZE=3; N4_SHRINK=4; N4_ITERS=[50,50,30,20]; NORM_PCTL=(0.5,99.5)
print('Casos:', CASOS)
print('Modalidad de segmentación:', MOD_WT.upper(), '| objetivo: tumor completo (seg>0)')

In [ ]:
# === Utilidades: carga, limpieza, semilla, métricas ===
def _extraer(cid, mod):
    suf=f'{cid}-{mod}.nii.gz'
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        m=next((n for n in zf.namelist() if n.endswith(suf)), None)
        if m is None: return None
        dest=os.path.join(TMP_DIR, os.path.basename(m))
        if not os.path.exists(dest):
            with zf.open(m) as s, open(dest,'wb') as d: shutil.copyfileobj(s,d)
    return dest

def cargar_np(cid, mod):
    p=_extraer(cid, mod)
    return nib.load(p).get_fdata(dtype=np.float32) if p else None
def cargar_seg(cid):
    p=_extraer(cid,'seg')
    return np.asarray(nib.load(p).dataobj).astype(np.int16) if p else None

def limpiar(vol):
    """Wiener (denoise) -> N4 (sesgo) -> normalización [0,1] dentro del cerebro."""
    brain = vol>0
    den = np.clip(np.nan_to_num(wiener(vol, mysize=WIENER_SIZE)),0,None).astype(np.float32)
    img = sitk.GetImageFromArray(den); mask=sitk.GetImageFromArray(brain.astype(np.uint8)); mask.CopyInformation(img)
    img_s=sitk.Shrink(img,[N4_SHRINK]*3); mask_s=sitk.Shrink(mask,[N4_SHRINK]*3)
    corr=sitk.N4BiasFieldCorrectionImageFilter(); corr.SetMaximumNumberOfIterations(N4_ITERS)
    corr.Execute(img_s, mask_s)
    n4=sitk.GetArrayFromImage(img/sitk.Exp(corr.GetLogBiasFieldAsImage(img))).astype(np.float32)
    out=np.zeros_like(n4); v=n4[brain]
    lo,hi=np.percentile(v,NORM_PCTL); hi=hi if hi>lo else lo+1
    out[brain]=np.clip((n4[brain]-lo)/(hi-lo),0,1)
    return out, brain

def centroide(mask):
    idx=np.argwhere(mask); return None if len(idx)==0 else tuple(int(round(c)) for c in idx.mean(0))
def keep_largest(m):
    lbl,n=ndi.label(m)
    if n==0: return m
    s=ndi.sum(np.ones_like(lbl),lbl,range(1,n+1)); return lbl==(np.argmax(s)+1)
def dice(a,b):
    a=a.astype(bool);b=b.astype(bool);s=a.sum()+b.sum(); return float(2*(a&b).sum()/s) if s else 0.0
def jaccard(a,b):
    a=a.astype(bool);b=b.astype(bool);u=(a|b).sum(); return float((a&b).sum()/u) if u else 0.0
def sensibilidad(p,g):
    p=p.astype(bool);g=g.astype(bool); return float((p&g).sum()/g.sum()) if g.sum() else 0.0
print('Utilidades listas.')

---
## Segmentación — 4 métodos clásicos (tumor completo en FLAIR)
Otsu multinivel (3 clases, clase más brillante), GMM (componente de mayor media), crecimiento de regiones (ConnectedThreshold sembrado en el centroide del tumor — semi-automático), y watershed con marcadores por intensidad.

In [ ]:
# === Métodos de segmentación (operan sobre FLAIR limpio [0,1]) ===
def m_otsu(flair, brain):
    v=flair[brain]
    if v.size<10: return np.zeros_like(flair,bool)
    thr=threshold_multiotsu(v, classes=3)
    return keep_largest((np.digitize(flair,thr)>=2) & brain)

def m_gmm(flair, brain, k=3):
    v=flair[brain].reshape(-1,1)
    if v.shape[0]<k: return np.zeros_like(flair,bool)
    sub=v if v.shape[0]<=50000 else v[np.random.default_rng(42).choice(v.shape[0],50000,replace=False)]
    g=GaussianMixture(n_components=k,random_state=42,max_iter=100).fit(sub)
    hi=int(np.argmax(g.means_.ravel()))
    pred=g.predict(v); m=np.zeros(flair.shape,bool); m[brain]=(pred==hi)
    return keep_largest(m)

def m_crecimiento(flair, brain, seed_zyx):
    if seed_zyx is None: return np.zeros_like(flair,bool)
    thr=threshold_multiotsu(flair[brain], classes=3)
    img=sitk.GetImageFromArray(flair)
    seed_xyz=(int(seed_zyx[2]),int(seed_zyx[1]),int(seed_zyx[0]))   # numpy (z,y,x) -> sitk (x,y,z)
    out=sitk.ConnectedThreshold(img, seedList=[seed_xyz], lower=float(thr[-1]), upper=float(flair.max()))
    return keep_largest(sitk.GetArrayFromImage(out).astype(bool) & brain)

def m_watershed(flair, brain):
    v=flair[brain]
    if v.size<10: return np.zeros_like(flair,bool)
    thr=threshold_multiotsu(v, classes=3)
    grad=sobel(flair)
    markers=np.zeros(flair.shape,np.int32)
    markers[(flair<=thr[0]) & brain]=1            # fondo / tejido sano
    markers[(flair>=thr[-1]) & brain]=2           # núcleo tumoral brillante
    ws=watershed(grad, markers, mask=brain)
    return keep_largest(ws==2)

def segmentar(metodo, flair, brain, seed_zyx):
    if metodo=='otsu':        return m_otsu(flair, brain)
    if metodo=='gmm':         return m_gmm(flair, brain)
    if metodo=='crecimiento': return m_crecimiento(flair, brain, seed_zyx)
    if metodo=='watershed':   return m_watershed(flair, brain)
    raise ValueError(metodo)
print('Métodos listos.')

In [ ]:
# === Ejecutar limpieza + segmentación + métricas sobre todos los casos ===
for met in METODOS: os.makedirs(os.path.join(SEG_DIR, met), exist_ok=True)
filas=[]; centroides={}
for k,cid in enumerate(CASOS,1):
    t0=time.time()
    flair_raw=cargar_np(cid, MOD_WT); seg=cargar_seg(cid)
    if flair_raw is None or seg is None:
        print('  faltan datos para', cid); continue
    gt=(seg>0)                                   # tumor completo
    flair, brain = limpiar(flair_raw)
    # guardar FLAIR limpio
    os.makedirs(os.path.join(LIMP_DIR,cid),exist_ok=True)
    np.save(os.path.join(LIMP_DIR,cid,f'{cid}-{MOD_WT}_limpio.npy'), flair.astype(np.float32))
    seed=centroide(gt); centroides[cid]=seed
    for met in METODOS:
        pred=segmentar(met, flair, brain, seed)
        _md=os.path.join(SEG_DIR,met); os.makedirs(_md,exist_ok=True)
        np.save(os.path.join(_md,f'{cid}.npy'), pred.astype(np.uint8))
        filas.append({'case_id':cid,'metodo':met,
                      'dice':round(dice(pred,gt),4),'jaccard':round(jaccard(pred,gt),4),
                      'sensibilidad':round(sensibilidad(pred,gt),4),
                      'vox_pred':int(pred.sum()),'vox_gt':int(gt.sum())})
    # limpiar temporales del caso
    for f in glob.glob(os.path.join(TMP_DIR,f'{cid}-*')): os.remove(f)
    gc.collect()
    print(f'  [{k}/{len(CASOS)}] {cid}  ({time.time()-t0:.0f}s)')
df_met=pd.DataFrame(filas)
df_met.to_csv(os.path.join(VIS_DIR,'metricas_segmentacion.csv'),index=False)
print('metricas_segmentacion.csv guardado.')
display(df_met)

In [ ]:
# === Tabla comparativa (resumen por método) ===
resumen=(df_met.groupby('metodo').agg(dice=('dice','mean'), jaccard=('jaccard','mean'),
         sensibilidad=('sensibilidad','mean')).sort_values('dice',ascending=False).reset_index())
resumen.to_csv(os.path.join(VIS_DIR,'resumen_metodos.csv'),index=False)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(resumen['metodo'], resumen['dice'])
ax.set_ylabel('Dice medio (tumor completo)'); ax.set_title('Comparación de métodos clásicos'); ax.grid(alpha=0.3,axis='y')
for i,v in enumerate(resumen['dice']): ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')
plt.tight_layout(); plt.savefig(os.path.join(VIS_DIR,'comparacion_metodos.png'),dpi=130,bbox_inches='tight'); plt.show()
display(resumen.round(3))

---
## Visualización — mosaicos de 3 vistas centrados en el tumor
Para cada caso demostrativo: cortes axial/sagital/coronal en el centroide del tumor (validado en 3D Slicer), FLAIR en gris con el ground truth y la predicción de cada método superpuestos (`origin='lower'`).

In [ ]:
# === Mosaicos 3 vistas (GT vs cada método) por caso ===
def _vistas(vol, c):
    return [('Axial', vol[c[0],:,:]), ('Sagital', vol[:,:,c[2]]), ('Coronal', vol[:,c[1],:])]
N_FIG=min(3, len(CASOS))
for cid in CASOS[:N_FIG]:
    flair=np.load(os.path.join(LIMP_DIR,cid,f'{cid}-{MOD_WT}_limpio.npy'))
    seg=cargar_seg(cid); gt=(seg>0); c=centroides.get(cid) or centroide(gt)
    if c is None: continue
    overlays=[('Ground truth', gt)]+[(met, np.load(os.path.join(SEG_DIR,met,f'{cid}.npy')).astype(bool)) for met in METODOS]
    fig,axes=plt.subplots(len(overlays),3,figsize=(10, 3.0*len(overlays)))
    for r,(nombre,mask) in enumerate(overlays):
        for col,(vt,sl) in enumerate(_vistas(flair,c)):
            ax=axes[r,col]; ax.imshow(sl,cmap='gray',origin='lower',aspect='auto')
            msl=_vistas(mask.astype(float),c)[col][1]
            ax.imshow(np.ma.masked_where(msl<0.5,msl),cmap='autumn',alpha=0.5,origin='lower',aspect='auto')
            ax.axis('off')
            if col==0: ax.set_ylabel(nombre,rotation=0,ha='right',va='center',fontsize=10)
            if r==0: ax.set_title(vt)
    fig.suptitle(f'{cid} — tumor completo (FLAIR) · GT vs métodos', y=1.0)
    plt.tight_layout(); plt.savefig(os.path.join(VIS_DIR,f'comparacion_{cid}.png'),dpi=120,bbox_inches='tight'); plt.show()
    for f in glob.glob(os.path.join(TMP_DIR,f'{cid}-*')): os.remove(f)
print('Mosaicos guardados en', VIS_DIR)

---
## Exportación 3D — mallas y nubes de puntos (GT vs mejor método)
A partir de las máscaras ya segmentadas: *marching cubes* → malla de superficie (OBJ, en mm vía el affine del volumen), nube de puntos de superficie (PLY) y un visor 3D interactivo (HTML) que compara el ground truth con el mejor método por Dice.

In [ ]:
# === Utilidades 3D (marching cubes -> OBJ / PLY) ===
from skimage import measure

DIR_3D = os.path.join(VIS_DIR, '3d'); os.makedirs(DIR_3D, exist_ok=True)
SIGMA_3D = 0.6   # suavizado previo a marching cubes (0 = sin suavizar)

def malla(mask, affine, sigma=SIGMA_3D):
    """Devuelve (vertices_mm, caras) por marching cubes. None si la máscara está vacía."""
    m = ndi.binary_fill_holes(keep_largest(mask.astype(bool)))
    if m.sum() < 10: return None, None
    vol = ndi.gaussian_filter(m.astype(np.float32), sigma) if sigma > 0 else m.astype(np.float32)
    if vol.max() < 0.5: return None, None
    verts, faces, _, _ = measure.marching_cubes(vol, level=0.5)
    world = (affine[:3,:3] @ verts.T).T + affine[:3,3]     # índices de voxel (i,j,k) -> mm
    return world, faces

def escribir_obj(path, verts, faces):
    with open(path, 'w') as f:
        f.write('# malla de superficie de la segmentacion (mm)\n')
        for v in verts: f.write('v %.3f %.3f %.3f\n' % (v[0], v[1], v[2]))
        for t in faces: f.write('f %d %d %d\n' % (t[0]+1, t[1]+1, t[2]+1))   # OBJ es 1-indexado

def nube_superficie(mask, affine):
    m = keep_largest(mask.astype(bool))
    surf = m & ~ndi.binary_erosion(m)
    idx = np.argwhere(surf).astype(float)
    return (affine[:3,:3] @ idx.T).T + affine[:3,3]

def escribir_ply(path, pts):
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\nelement vertex %d\n' % len(pts))
        f.write('property float x\nproperty float y\nproperty float z\nend_header\n')
        for p in pts: f.write('%.3f %.3f %.3f\n' % (p[0], p[1], p[2]))
print('Utilidades 3D listas. Salidas ->', DIR_3D)

In [ ]:
# === Exportar 3D: GT + mejor método (por Dice) para los casos demostrativos ===
try:
    import plotly.graph_objects as go; _PLOTLY = True
except Exception:
    _PLOTLY = False

# mejor método por Dice medio (de resumen_metodos.csv o de df_met en memoria)
try:
    _res = pd.read_csv(os.path.join(VIS_DIR, 'resumen_metodos.csv'))
    MEJOR = _res.sort_values('dice', ascending=False).iloc[0]['metodo']
except Exception:
    MEJOR = df_met.groupby('metodo')['dice'].mean().idxmax()
print('Mejor método por Dice:', MEJOR)

N_3D = min(3, len(CASOS))
for cid in CASOS[:N_3D]:
    p_seg = _extraer(cid, 'seg')
    if not p_seg: print('  sin seg para', cid); continue
    im = nib.load(p_seg); affine = im.affine
    gt = np.asarray(im.dataobj).astype(np.int16) > 0
    pred_path = os.path.join(SEG_DIR, MEJOR, f'{cid}.npy')
    pred = np.load(pred_path).astype(bool) if os.path.exists(pred_path) else np.zeros_like(gt)

    piezas = {'GT': gt, MEJOR: pred}
    mallas = {}
    for nombre, mask in piezas.items():
        v, fcs = malla(mask, affine)
        if v is None:
            print(f'  {cid} · {nombre}: máscara vacía, sin malla'); continue
        escribir_obj(os.path.join(DIR_3D, f'{cid}_{nombre}.obj'), v, fcs)
        escribir_ply(os.path.join(DIR_3D, f'{cid}_{nombre}_nube.ply'), nube_superficie(mask, affine))
        mallas[nombre] = (v, fcs)

    # visor 3D interactivo (GT verde vs método rojo)
    if _PLOTLY and mallas:
        fig = go.Figure()
        colores = {'GT': '#22C55E', MEJOR: '#EF4444'}
        for nombre, (v, fcs) in mallas.items():
            fig.add_trace(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2],
                                    i=fcs[:,0], j=fcs[:,1], k=fcs[:,2],
                                    color=colores.get(nombre, '#3B82F6'), opacity=0.5, name=nombre, showscale=False))
        fig.update_layout(title=f'{cid} — GT (verde) vs {MEJOR} (rojo)',
                          scene=dict(aspectmode='data'), width=820, height=620)
        fig.write_html(os.path.join(DIR_3D, f'{cid}_preview3d.html'))
    print('  3D exportado:', cid, '->', list(mallas.keys()))
    for f in glob.glob(os.path.join(TMP_DIR, f'{cid}-*')): os.remove(f)

print('Mallas OBJ + nubes PLY + previews HTML en', DIR_3D)

---
## Reporte HTML del proyecto

In [ ]:
# === Reporte HTML (informe del proyecto, estilo académico) ===
import base64
try:
    import plotly.graph_objects as go
    import plotly.io as pio
    _PLOTLY = True
except Exception:
    _PLOTLY = False

def _png_b64(path):
    if not os.path.exists(path): return None
    return 'data:image/png;base64,' + base64.b64encode(open(path, 'rb').read()).decode()

# --- métricas ---
res = pd.read_csv(os.path.join(VIS_DIR, 'resumen_metodos.csv')).sort_values('dice', ascending=False)
det = pd.read_csv(os.path.join(VIS_DIR, 'metricas_segmentacion.csv'))
mejor_met = res.iloc[0]['metodo']; mejor_dice = float(res.iloc[0]['dice'])
peor_met  = res.iloc[-1]['metodo']; peor_dice = float(res.iloc[-1]['dice'])
n_casos = int(det['case_id'].nunique()); n_met = int(det['metodo'].nunique())

CSS = """<style>
:root{--azul:#1f77b4;--naranja:#ff7f0e;--morado:#9467bd;--verde:#2ca02c;--rojo:#d62728;
 --tinta:#1f2a37;--suave:#5b6b7c;--linea:#e3e8ef;--fondo:#f6f8fb}
*{box-sizing:border-box}
body{font-family:'Segoe UI',system-ui,Arial,sans-serif;margin:0;color:var(--tinta);background:var(--fondo);line-height:1.55}
.wrap{max-width:1180px;margin:0 auto;padding:0 22px 60px}
header.hero{background:linear-gradient(120deg,#1e3c72,#2a5298 60%,#3a7bd5);color:#fff;padding:38px 22px 30px}
header.hero h1{margin:0 0 6px;font-size:26px} header.hero p{margin:0;color:#dce6f7;max-width:860px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:20px 0}
.kpi{background:#fff;border-radius:12px;padding:14px 16px;box-shadow:0 1px 3px rgba(20,40,80,.08);border-top:4px solid var(--azul)}
.kpi .v{font-size:23px;font-weight:700} .kpi .l{color:var(--suave);font-size:13px}
.kpi.l2{border-color:var(--morado)} .kpi.l3{border-color:var(--verde)} .kpi.l4{border-color:var(--naranja)}
h2{margin:36px 0 6px;font-size:20px;border-left:5px solid var(--azul);padding-left:12px}
h3{margin:16px 0 6px;font-size:15px} p{max-width:980px}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:14px 0}
.card{background:#fff;border-radius:12px;padding:14px 16px;box-shadow:0 1px 3px rgba(20,40,80,.07);border:1px solid var(--linea)}
.card.m{border-top:5px solid var(--azul)} .card.pm{border-left:4px solid var(--naranja);background:#fff8f1}
.fig{background:#fff;border:1px solid var(--linea);border-radius:12px;padding:8px;margin:12px 0}
table{border-collapse:collapse;margin:8px 0;width:100%;background:#fff;border-radius:10px;overflow:hidden;box-shadow:0 1px 3px rgba(20,40,80,.06)}
th,td{border-bottom:1px solid var(--linea);padding:7px 12px;text-align:right;font-size:13px}
th{background:#eef2f8;color:#2a3f5f} td:first-child,th:first-child{text-align:left}
tbody tr:nth-child(even){background:#fafbfd}
.note{color:var(--suave);font-size:13px} select{padding:6px 10px;border-radius:8px;border:1px solid var(--linea);font-size:14px}
</style>"""

def _tabla(df): return df.round(3).to_html(index=False, border=0)

img_barras = _png_b64(os.path.join(VIS_DIR, 'comparacion_metodos.png'))

# --- mosaicos 2D ---
mosaicos = ''
for cid in CASOS:
    b = _png_b64(os.path.join(VIS_DIR, f'comparacion_{cid}.png'))
    if b: mosaicos += f'<h3>{cid}</h3><div class="fig"><img src="{b}" style="width:100%"></div>'

# --- 3D interactivo (GT vs mejor método, selector por caso) ---
sel_html = ''; divs3d = ''; script3d = ''
if _PLOTLY:
    figs, opts = [], []
    for i, cid in enumerate(CASOS):
        p_seg = _extraer(cid, 'seg')
        if not p_seg: continue
        im = nib.load(p_seg); affine = im.affine
        gt = np.asarray(im.dataobj).astype(np.int16) > 0
        pm = os.path.join(SEG_DIR, mejor_met, f'{cid}.npy')
        pred = np.load(pm).astype(bool) if os.path.exists(pm) else np.zeros_like(gt)
        fig = go.Figure()
        for nombre, mask, color in [('GT', gt, '#2ca02c'), (mejor_met, pred, '#d62728')]:
            v, fcs = malla(mask, affine)
            if v is None: continue
            fig.add_trace(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=fcs[:,0], j=fcs[:,1], k=fcs[:,2],
                                    color=color, opacity=0.5, name=nombre, showscale=False))
        fig.update_layout(scene=dict(aspectmode='data'), height=560, margin=dict(l=0, r=0, t=8, b=0),
                          legend=dict(orientation='h'))
        div = pio.to_html(fig, full_html=False, include_plotlyjs=('cdn' if not figs else False), default_height='560px')
        figs.append(f'<div class="vol3d" data-k="{i}" style="display:{"block" if not opts else "none"}">{div}</div>')
        opts.append(f'<option value="{i}">{cid}</option>')
        for f in glob.glob(os.path.join(TMP_DIR, f'{cid}-*')): os.remove(f)
    sel_html = f'<p>Seleccione un caso: <select id="sel3d" onchange="ver3d(this.value)">{"".join(opts)}</select></p>'
    divs3d = ''.join(figs)
    script3d = '<script>function ver3d(i){document.querySelectorAll(".vol3d").forEach(function(d){d.style.display=(d.getAttribute("data-k")===i)?"block":"none";});window.dispatchEvent(new Event("resize"));}</script>'
else:
    divs3d = '<p class="note">Plotly no disponible: ver los archivos <code>*_preview3d.html</code> en <code>visualizacion/3d/</code>.</p>'

# --- bloques de texto ---
kpis = ('<div class="kpis">'
        f'<div class="kpi"><div class="v">{mejor_dice:.3f}</div><div class="l">Mejor Dice &mdash; {mejor_met}</div></div>'
        f'<div class="kpi l2"><div class="v">{n_casos}</div><div class="l">Casos analizados</div></div>'
        f'<div class="kpi l3"><div class="v">{n_met}</div><div class="l">M&eacute;todos cl&aacute;sicos</div></div>'
        '<div class="kpi l4"><div class="v">FLAIR</div><div class="l">Tumor completo</div></div></div>')

resumen = (f'<p>Se segment&oacute; el <b>tumor completo</b> (whole tumor) en la modalidad <b>FLAIR</b> de {n_casos} '
           'casos de BraTS 2024 GLI con cuatro m&eacute;todos cl&aacute;sicos, sin aprendizaje profundo. '
           f'El mejor desempe&ntilde;o medio lo obtuvo <b>{mejor_met}</b> (Dice {mejor_dice:.3f}); el m&aacute;s bajo fue '
           f'<b>{peor_met}</b> (Dice {peor_dice:.3f}). Los valores moderados son esperables y constituyen el hallazgo '
           'central: los m&eacute;todos cl&aacute;sicos basados en intensidad son insuficientes para el tumor completo post-tratamiento.</p>')

metodos_cards = ('<div class="grid2">'
 '<div class="card m"><h4>Otsu multinivel (3 clases)</h4><p class="note">Umbralizaci&oacute;n global; clase m&aacute;s brillante = tumor. Falla si el histograma no es claramente multimodal.</p></div>'
 '<div class="card m"><h4>GMM (mixtura de gaussianas)</h4><p class="note">Modela la intensidad como mezcla de gaussianas; tumor = componente de mayor media. M&aacute;s flexible que Otsu, pero igual solo intensidad.</p></div>'
 '<div class="card m"><h4>Crecimiento de regiones</h4><p class="note">ConnectedThreshold sembrado en el centroide del tumor (semi-autom&aacute;tico). Sensible a la semilla: si cae en la cavidad oscura, captura poco.</p></div>'
 '<div class="card m"><h4>Watershed (marcadores)</h4><p class="note">Inundaci&oacute;n sobre el gradiente con marcadores de intensidad (sano vs n&uacute;cleo brillante). Tiende a sobre/sub-segmentar en bordes difusos.</p></div>'
 '</div>')

discusion = ('<div class="card pm"><p>El <b>tumor completo</b> incluye la <b>cavidad de resecci&oacute;n</b>, que es '
 '<b>oscura</b> en FLAIR, adem&aacute;s de tejido heterog&eacute;neo y edema con bordes difusos. Por eso la heur&iacute;stica '
 '"lo m&aacute;s brillante = tumor" (Otsu, GMM, watershed) pierde partes oscuras, y el crecimiento de regiones falla '
 'cuando la semilla (centroide) cae dentro de la cavidad. Es justamente la limitaci&oacute;n de los m&eacute;todos '
 'cl&aacute;sicos frente a deep learning que el proyecto busca evidenciar.</p></div>')

repro = (f'<p class="note">Modalidad: FLAIR (t2f) &middot; objetivo: tumor completo (seg&gt;0) &middot; '
 f'm&eacute;todos: {", ".join(res["metodo"].tolist())} &middot; '
 f'limpieza: Wiener(size={WIENER_SIZE}) &rarr; N4(shrink={N4_SHRINK}) &rarr; normalizaci&oacute;n percentiles {NORM_PCTL} &middot; '
 'semilla: centroide del ground truth (semi-autom&aacute;tico).</p>')

html = ('<!doctype html><html lang="es"><head><meta charset="utf-8">'
 '<title>Segmentaci\u00f3n BraTS 2024 GLI \u2014 reporte</title>' + CSS + '</head><body>'
 '<header class="hero"><div class="wrap"><h1>Segmentaci&oacute;n cl&aacute;sica de tumores &mdash; BraTS 2024 GLI</h1>'
 '<p>Tumor completo en FLAIR &middot; 4 m&eacute;todos cl&aacute;sicos &middot; m&eacute;tricas Dice/Jaccard, reconstrucci&oacute;n 3D y an&aacute;lisis de limitaciones.</p></div></header>'
 '<div class="wrap">' + kpis +
 '<h2>1. Resumen ejecutivo</h2>' + resumen +
 '<h2>2. M&eacute;todos cl&aacute;sicos</h2>' + metodos_cards +
 '<h2>3. M&eacute;tricas &mdash; comparaci&oacute;n</h2>' + (f'<div class="fig"><img src="{img_barras}" style="width:100%"></div>' if img_barras else '') +
 '<h3>Resumen por m&eacute;todo</h3>' + _tabla(res) +
 '<h3>Detalle por caso</h3>' + _tabla(det) +
 '<h2>4. Mosaicos 2D &mdash; GT vs m&eacute;todos</h2>' + (mosaicos or '<p class="note">Sin mosaicos.</p>') +
 '<h2>5. Reconstrucci&oacute;n 3D</h2>' + sel_html + divs3d +
 '<h2>6. Discusi&oacute;n y limitaciones</h2>' + discusion +
 '<h2>7. Reproducibilidad</h2>' + repro +
 '</div>' + script3d + '</body></html>')

out_report = os.path.join(VIS_DIR, 'report.html')
with open(out_report, 'w', encoding='utf-8') as f:
    f.write(html)
print('Reporte HTML guardado en', out_report, '|', round(os.path.getsize(out_report)/1e6, 2), 'MB')


In [ ]:
# === Empaquetar y descargar salidas ===
try:
    from google.colab import files; _COLAB=True
except Exception:
    _COLAB=False
# copiar salidas locales a Drive (best-effort) para que queden en final-project/
if DRIVE_BASE:
    for sub in ['limpieza','segmentacion','visualizacion']:
        try:
            shutil.copytree(os.path.join(OUT_BASE,sub), os.path.join(DRIVE_BASE,sub), dirs_exist_ok=True)
        except Exception as e:
            print('aviso: no se pudo copiar a Drive', sub, '->', e)
    print('Salidas copiadas a', DRIVE_BASE)
zip_out='/content/segmentacion_visualizacion_salidas.zip' if EN_COLAB else 'seg_vis_salidas.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as zf:
    for base in [LIMP_DIR, SEG_DIR, VIS_DIR]:
        for root,_,fnames in os.walk(base):
            for fn in fnames:
                fp=os.path.join(root,fn); zf.write(fp, os.path.relpath(fp,OUT_BASE))
print('ZIP:', zip_out, round(os.path.getsize(zip_out)/1e6,2),'MB')
if _COLAB: files.download(zip_out)